# **Imports**

In [ ]:
import pandas as pd
import networkx as nx
import community
import random
import csv
import re
import matplotlib.pyplot as plt
from collections import Counter
import itertools 

In [89]:
#I aim to run community detection on only the comments as they are more insightful than posts
reddit = pd.read_csv("Reddit/clean_40k_comments.csv")
youtube = pd.read_csv("Anime_Data_Youtube_CLEANED.csv")

In [90]:
reddit.head()

,post_id,comment_id,body,score,author,created_utc,cleaned_body
0,1kt8o3b,mtry4sy,TCB's translation is a bit off for Imu's dialo...,1212,adnaphsaka,2025-05-23 03:38:25,tcbs translation bit imu dialogue imu say nani...
1,1kt8o3b,mtrrd7o,It would be classic Oda to finally cut back to...,1732,Brookboy,2025-05-23 02:52:45,would classic oda finally cut back luffys group
2,1kt8o3b,mts2xpb,https://preview.redd.it/o0s6ww1zig2f1.jpeg?wid...,1162,Blue-Diamond-Enjoyer,2025-05-23 04:13:34,holy fuck
3,1kt8o3b,mtrp7pf,Everyone: Elbaf is gonna be Usopp’s arc\n\nOda...,3062,WeAreHereWithAll,2025-05-23 02:39:18,everyone elbaf gon usopps arc oda brook might ...
4,1kt8o3b,mtrp2v6,what \n\n\nWHAT\n\n\nASSUMING DIRECT CONTROL,1306,see_mohn,2025-05-23 02:38:29,assuming direct control


In [91]:
reddit_comments = reddit['cleaned_body'].dropna().tolist()
print(reddit_comments[:10])

['tcbs translation bit imu dialogue imu say nani guzu guzu shiteru seichi ima umi kakujitsu senryoku yosurunoda misete yaru kami shihai taking long holy land sea flame right iwe definitely need military power show reign god note imu us word rulereign shihai luffy used told rayleigh going rule anything', 'would classic oda finally cut back luffys group', 'holy fuck', 'everyone elbaf gon usopps arc oda brook might dad god knight sister mother fucker chapter fucking cracked', 'assuming direct control', 'imu committed silhouette bit even gunkos already wrappedup face get silhouetted hell good guy get situation wtf', 'dragon child parent greatest weakness hell fine gon fall right luffy others probably', 'brook', 'garp said game get stabbed koby gyaban said he rusty get stabbed son whitebeard sick dying stabbed son kuma fly egghead tank stab bonney oden prime got blindsided bonked kaido son cent every parental figure getting hole put losing limb future generation every judge yassop there zef

In [92]:
youtube.head()

,Timestamp,ChannelID,Username,VideoID,Comment,Date,Likes,subscriberCount,tokens
0,2024-01-29T17:30:55Z,UCv3LuuW8auzYw1SdLmiLRAw,@TotallyNotMark,S63luLJUTmo,Who's your favourite Straw Hat pirate and why?,2024-01-29T17:30:55Z,819,1010000.0,"['favourite', 'straw', 'hat', 'pirate']"
1,2024-01-29T17:32:22Z,UCXFcawqqFE3yGO6zFM_CFVg,@ectospassmta8493,S63luLJUTmo,"I love Usopp, I've always been a fan of underd...",2024-01-29T17:32:22Z,129,23.0,"['love', 'usopp', 'always', 'fan', 'underdog',..."
2,2024-01-29T17:32:25Z,UCVpE0TaPyuxzUYkJs9_Rdcw,@MrAwesomeVVV,S63luLJUTmo,I like franky because he’s such a silly guy :3,2024-01-29T17:32:25Z,81,72.0,"['like', 'franky', 'silly', 'guy']"
3,2024-01-29T17:33:19Z,UCMCaUbFLqbObV-BYuAZfT5w,@iamsam-jm5ji,S63luLJUTmo,I’ve gotten to really love Jinbe and how long ...,2024-01-29T17:33:19Z,47,29.0,"['gotten', 'really', 'love', 'jinbe', 'long', ..."
4,2024-01-29T17:34:57Z,UCsbfwXu7RwhKDAZZ5f5f8Bg,@HGmolotov,S63luLJUTmo,"I like zoro, because in spite of the yelling a...",2024-01-29T17:34:57Z,51,9.0,"['like', 'zoro', 'spite', 'yelling', 'simple',..."


In [93]:
yt_comments = youtube['Comment'].dropna().tolist()
print(yt_comments[:10])

["Who's your favourite Straw Hat pirate and why?", "I love Usopp, I've always been a fan of underdog characters that fight the good fight no matter how scared or weak they may be. I love Krillin for the same reason.", 'I like franky because he’s such a silly guy :3', 'I’ve gotten to really love Jinbe and how long it took him to become a straw hat. He really had to fight for it', "I like zoro, because in spite of the yelling and how simple and driven he is, he's written very subtly, plus he has some of my favourite crew dynamics", 'Probably Robin. She has this very smart and dignified air about her, but on the inside she’s rather similar to Luffy. This makes for a pretty fun character because she’s both very intelligent but also has her own brand of silliness', "Jinbei cuz i want him to be my dad :0  he's so lovely and i wanna hug him", "Atm, my favourite is Luffy, I love how he mainly listens to those he interacts with, and then speaks his mind no holds barred, he's also super funny.",

In [94]:
print(f"Youtube comments length: {len(yt_comments)}")
print(f"Reddit comments length: {len(reddit_comments)}")

#Put them in the same file for convenience when implementing the community detection
comments = yt_comments + reddit_comments
print(f"Comments length: {len(comments)}")

Youtube comments length: 49752
Reddit comments length: 38195
Comments length: 87947


# **Preprocessing**

In [95]:
#Define a function that will handle case-sensitive
def extract_character_mentions(text, char_list):
    text = text.lower()
    text = re.sub(r'[’‘”“]', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+|u/\w+', '', text)
    tokens = text.split()

    return list(set([char for char in char_list if char in tokens]))

# **Character Co-occurrence Count**

In [96]:
#Storing characters in a list - These will be the nodes
straw_hats = ['luffy', 'zoro', 'sanji', 'nami', 'chopper', 'usopp', 'jinbei', 'robin', 'franky', 'brook']

pair_counter = Counter()


for comment in comments:
    mentioned = extract_character_mentions(comment, straw_hats)
    if len(mentioned) >= 2:
      pairs = itertools.combinations(sorted(mentioned), 2)
      pair_counter.update(pairs)

print(pair_counter.most_common)

<bound method Counter.most_common of Counter({('luffy', 'zoro'): 1233, ('luffy', 'nami'): 1196, ('sanji', 'zoro'): 1046, ('luffy', 'sanji'): 916, ('luffy', 'robin'): 587, ('nami', 'sanji'): 518, ('nami', 'robin'): 504, ('luffy', 'usopp'): 501, ('robin', 'sanji'): 479, ('nami', 'zoro'): 410, ('sanji', 'usopp'): 350, ('brook', 'luffy'): 342, ('brook', 'robin'): 340, ('chopper', 'luffy'): 332, ('franky', 'robin'): 331, ('chopper', 'robin'): 326, ('robin', 'zoro'): 322, ('nami', 'usopp'): 314, ('usopp', 'zoro'): 300, ('chopper', 'sanji'): 275, ('brook', 'sanji'): 274, ('chopper', 'nami'): 266, ('franky', 'luffy'): 266, ('brook', 'franky'): 254, ('brook', 'chopper'): 239, ('robin', 'usopp'): 230, ('chopper', 'zoro'): 229, ('brook', 'zoro'): 226, ('chopper', 'franky'): 219, ('franky', 'sanji'): 215, ('chopper', 'usopp'): 212, ('franky', 'zoro'): 195, ('brook', 'nami'): 194, ('franky', 'usopp'): 193, ('franky', 'nami'): 187, ('brook', 'usopp'): 165, ('jinbei', 'luffy'): 106, ('jinbei', 'sanji

Construct Graph

In [101]:
G = nx.Graph()
G.add_nodes_from(straw_hats)

print(f"Number of nodes: {G.number_of_nodes()}")

Number of nodes: 10
